# 02 · Data Cleaning, Preprocessing & Leakage Prevention

**Owner:** Riya (Model Lead)

```
raw CSV ─► clean (TotalCharges, SeniorCitizen, Churn→0/1) ─► stratified split 60/20/20
        ─► fit StandardScaler + OneHotEncoder on TRAIN only ─► transform val / test
        ─► save scaler.pkl · encoder.pkl · feature_columns.pkl · splits.npz · sample_demo.csv
```

> **Viva Q1 — "Why did you fit the scaler only on the training set?"**
> The validation and test sets simulate *future, unseen* customers. If `StandardScaler` computed μ and σ (or the encoder its category list) using those rows, their information would leak into training and every reported metric would be optimistically biased. So `.fit()` sees **only `X_train`**; validation, test and every CSV uploaded to the web app get `.transform()` with the training statistics.

In [1]:
# ── Environment setup: works on Google Colab (Drive) AND locally (VS Code / Jupyter) ──
import os, sys

try:
    from google.colab import drive  # only importable inside a Colab runtime
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/telco-churn-dashboard'
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    _cwd = os.path.abspath(os.getcwd())
    PROJECT_ROOT = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'notebooks' else _cwd

os.makedirs(PROJECT_ROOT, exist_ok=True)
os.chdir(PROJECT_ROOT)                      # every relative path is now project-relative
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)        # lets us `import src...`
print('Project root:', PROJECT_ROOT, '| running on Colab:', IN_COLAB)

Project root: /Users/friday/Desktop/nndl/telco-churn-dashboard | running on Colab: False


In [2]:
# Colab already ships torch, numpy, pandas, scikit-learn, matplotlib and seaborn.
# Install the extras once per Colab session (skipped automatically when running locally).
if IN_COLAB:
    %pip install -q imbalanced-learn shap fastapi uvicorn jinja2 python-multipart joblib

In [3]:
import numpy as np
import pandas as pd

from src import config
from src.data_preprocessing import (
    CATEGORICAL_COLS, ID_COL, NUMERIC_COLS, RAW_FEATURE_COLS, TARGET_COL,
    assert_disjoint_splits, clean_dataframe, fit_preprocessors, get_feature_names,
    load_preprocessors, load_raw_data, load_splits, save_preprocessors, save_splits,
    split_data, transform_features,
)

config.set_seed(config.SEED)   # random.seed(42), np.random.seed(42), torch.manual_seed(42)
config.ensure_dirs()

## 1 · Load and clean

In [4]:
raw_df = load_raw_data(config.RAW_DATA_PATH)
clean_df, report = clean_dataframe(raw_df, require_target=True)

print(f'Rows in: {report.n_input_rows:,}  ->  rows out: {report.n_output_rows:,}')
print(f'TotalCharges blanks filled with 0.0 (all have tenure == 0): {report.total_charges_blank_filled}')
print('Cleaning warnings:', report.warnings or 'none')

blank_ids = raw_df.loc[raw_df['TotalCharges'].astype(str).str.strip() == '', ID_COL]
clean_df.loc[clean_df[ID_COL].isin(blank_ids), [ID_COL, 'tenure', 'MonthlyCharges', 'TotalCharges', TARGET_COL]]

Rows in: 7,043  ->  rows out: 7,043
TotalCharges blanks filled with 0.0 (all have tenure == 0): 11
Cleaning warnings: none


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,0.0,0
753,3115-CZMZD,0,20.25,0.0,0
936,5709-LVOEQ,0,80.85,0.0,0
1082,4367-NUYAO,0,25.75,0.0,0
1340,1371-DWPAZ,0,56.05,0.0,0
3331,7644-OMVMY,0,19.85,0.0,0
3826,3213-VVOLG,0,25.35,0.0,0
4380,2520-SGTTA,0,20.00,0.0,0
5218,2923-ARZLG,0,19.70,0.0,0
6670,4075-WKNIU,0,73.35,0.0,0


**Justification for 0.0.** A blank `TotalCharges` occurs only when `tenure == 0` — the customer joined this month and has not yet been billed, so their cumulative charges really are zero. Mean or median imputation would fabricate a billing history.

`customerID` is **kept** in the frame (the dashboard needs it to identify customers) but it is **not** one of the 19 model inputs:

In [5]:
assert ID_COL not in RAW_FEATURE_COLS and TARGET_COL not in RAW_FEATURE_COLS
print(f'{len(RAW_FEATURE_COLS)} raw model inputs:', RAW_FEATURE_COLS)
print('Target encoding (Churn):', clean_df[TARGET_COL].value_counts().sort_index().to_dict(), ' (1 = Yes, 0 = No)')

19 raw model inputs: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']
Target encoding (Churn): {0: 5174, 1: 1869}  (1 = Yes, 0 = No)


## 2 · Stratified 60 / 20 / 20 split
Two calls to `train_test_split(stratify=y, random_state=42)`: first 80/20 (train+val vs **test**), then 75/25 of the 80 % (**train** vs **val**) → 60/20/20. Stratification keeps the churn rate identical in every split.

In [6]:
train_df, val_df, test_df = split_data(clean_df, val_size=config.VAL_SIZE,
                                       test_size=config.TEST_SIZE, seed=config.SEED)
split_summary = pd.DataFrame({
    'rows': [len(d) for d in (train_df, val_df, test_df)],
    'churners': [int(d[TARGET_COL].sum()) for d in (train_df, val_df, test_df)],
}, index=['train', 'validation', 'test'])
split_summary['share_%'] = (100 * split_summary['rows'] / len(clean_df)).round(1)
split_summary['churn_rate_%'] = (100 * split_summary['churners'] / split_summary['rows']).round(2)

assert_disjoint_splits(train_df, val_df, test_df)   # raises if any customer is in two splits
print('No customer appears in more than one split.')
split_summary

No customer appears in more than one split.


,rows,churners,share_%,churn_rate_%
train,4225,1121,60.0,26.53
validation,1409,374,20.0,26.54
test,1409,374,20.0,26.54


## 3 · Fit the scaler and encoder on the TRAINING split only
* `StandardScaler` on `tenure`, `MonthlyCharges`, `TotalCharges`: $z = (x-\mu_{train})/\sigma_{train}$.
* `OneHotEncoder(handle_unknown="ignore", drop="if_binary")` on the 16 categorical columns: Yes/No columns become one 0/1 feature; multi-level columns keep every level; unseen categories at inference become all-zeros instead of crashing the app.

In [7]:
scaler, encoder = fit_preprocessors(train_df)             # .fit() sees ONLY training rows

X_train = transform_features(train_df, scaler, encoder)
X_val   = transform_features(val_df,   scaler, encoder)   # .transform() only
X_test  = transform_features(test_df,  scaler, encoder)   # .transform() only
y_train, y_val, y_test = (d[TARGET_COL].to_numpy() for d in (train_df, val_df, test_df))
feature_columns = get_feature_names(encoder)

print('X_train', X_train.shape, '| X_val', X_val.shape, '| X_test', X_test.shape)
print(f'{len(feature_columns)} encoded features:')
print(feature_columns)

X_train (4225, 40) | X_val (1409, 40) | X_test (1409, 40)
40 encoded features:
['tenure', 'MonthlyCharges', 'TotalCharges', 'gender_Male', 'SeniorCitizen_Yes', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credit card (a

**Evidence of no leakage.** After scaling, the *training* columns have mean exactly 0 and std exactly 1. Validation and test are close but **not** exactly 0/1 — proof that their own statistics were never used.

In [8]:
print('Scaler learned  mu    =', dict(zip(NUMERIC_COLS, scaler.mean_.round(2))))
print('                sigma =', dict(zip(NUMERIC_COLS, scaler.scale_.round(2))))
pd.DataFrame({
    'train mean': X_train[:, :3].mean(0), 'train std': X_train[:, :3].std(0),
    'val mean': X_val[:, :3].mean(0), 'val std': X_val[:, :3].std(0),
    'test mean': X_test[:, :3].mean(0), 'test std': X_test[:, :3].std(0),
}, index=NUMERIC_COLS).round(4)

Scaler learned  mu    = {'tenure': np.float64(32.29), 'MonthlyCharges': np.float64(64.78), 'TotalCharges': np.float64(2284.04)}
                sigma = {'tenure': np.float64(24.67), 'MonthlyCharges': np.float64(29.99), 'TotalCharges': np.float64(2280.06)}


,train mean,train std,val mean,val std,test mean,test std
tenure,0.0,1.0,0.0309,0.9836,-0.0154,0.9940
MonthlyCharges,0.0,1.0,0.0205,1.0197,-0.0229,0.9967
TotalCharges,-0.0,1.0,0.0268,0.9979,-0.0363,0.9714


In [9]:
pd.Series({col: list(cats) for col, cats in zip(CATEGORICAL_COLS, encoder.categories_)},
          name='categories learned from TRAIN').to_frame()

,categories learned from TRAIN
gender,"[Female, Male]"
SeniorCitizen,"[No, Yes]"
Partner,"[No, Yes]"
Dependents,"[No, Yes]"
PhoneService,"[No, Yes]"
MultipleLines,"[No, No phone service, Yes]"
InternetService,"[DSL, Fiber optic, No]"
OnlineSecurity,"[No, No internet service, Yes]"
OnlineBackup,"[No, No internet service, Yes]"
DeviceProtection,"[No, No internet service, Yes]"


## 4 · Save the artefacts
* `models/scaler.pkl`, `models/encoder.pkl`, `models/feature_columns.pkl` — used by the inference API.
* `data/processed/splits.npz` — the encoded arrays, so notebooks 03–07 use *identical* splits.
* `data/processed/{train,val,test}_clean.csv` — human-readable cleaned splits.

In [10]:
import joblib

save_preprocessors(scaler, encoder, feature_columns, config.MODELS_DIR)
# (equivalent to: joblib.dump(scaler, 'models/scaler.pkl'); joblib.dump(encoder, 'models/encoder.pkl');
#                 joblib.dump(feature_columns, 'models/feature_columns.pkl'))

splits_path = save_splits({
    'train': (X_train, y_train, train_df[ID_COL]),
    'val':   (X_val,   y_val,   val_df[ID_COL]),
    'test':  (X_test,  y_test,  test_df[ID_COL]),
}, feature_columns)
for name, frame in [('train', train_df), ('val', val_df), ('test', test_df)]:
    frame.to_csv(config.PROCESSED_DIR / f'{name}_clean.csv', index=False)

for p in [config.SCALER_PATH, config.ENCODER_PATH, config.FEATURE_COLUMNS_PATH, splits_path]:
    print(f'saved {p.relative_to(config.PROJECT_ROOT)}  ({p.stat().st_size / 1024:.1f} KB)')

saved models/scaler.pkl  (0.6 KB)
saved models/encoder.pkl  (4.9 KB)
saved models/feature_columns.pkl  (0.9 KB)
saved data/processed/splits.npz  (182.8 KB)


## 5 · Ready-to-upload demo CSV (`data/sample_demo.csv`)
50 random customers from the **raw, uncleaned test split** (so the app's cleaning path is exercised exactly as with a real upload — e.g. `TotalCharges` stays a string), with the ground-truth `Churn` column **removed** and `customerID` kept for display. The demo only *displays* predictions; no metrics are computed from it, so the single-test-evaluation rule is respected.

In [11]:
test_raw_df = raw_df.loc[test_df.index]                 # untouched raw rows of the TEST split
demo_df = test_raw_df.sample(n=50, random_state=config.SEED).copy()
demo_df = demo_df.drop(columns=[TARGET_COL], errors='ignore')   # never ship the label
demo_df.to_csv(config.SAMPLE_DEMO_PATH, index=False)

print('sample_demo.csv:', demo_df.shape, '| contains Churn column?', TARGET_COL in demo_df.columns)
demo_df.head()

sample_demo.csv: (50, 20) | contains Churn column? False


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
5902,7054-ENNGU,Female,0,Yes,No,9,Yes,No,DSL,No,Yes,No,No,No,No,Month-to-month,No,Mailed check,50.85,466.6
5415,1310-QRITU,Female,0,No,No,18,Yes,Yes,DSL,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,50.30,913.3
2874,8347-GDTMP,Female,0,Yes,No,64,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Credit card (automatic),19.45,1225.65
2686,4283-FUTGF,Male,1,No,No,19,Yes,Yes,Fiber optic,No,No,Yes,No,No,No,Month-to-month,Yes,Electronic check,79.85,1471.75
700,1525-LNLOJ,Male,0,Yes,Yes,66,Yes,Yes,DSL,No,No,Yes,No,Yes,No,Two year,Yes,Bank transfer (automatic),63.30,4189.7


## 6 · Round-trip check

In [12]:
s2, e2, f2 = load_preprocessors(config.MODELS_DIR)
assert f2 == feature_columns
assert np.allclose(transform_features(val_df, s2, e2), X_val)

dev = load_splits()                       # default: include_test=False
assert dev.X_test is None                 # the test split stays hidden during development
print('Reloaded artefacts reproduce X_val exactly; the test split is only loadable with include_test=True.')

Reloaded artefacts reproduce X_val exactly; the test split is only loadable with include_test=True.


### Summary
| Step | Result |
|---|---|
| Blank `TotalCharges` | 11 rows, all `tenure == 0` → filled with 0.0 |
| Split | 4,225 / 1,409 / 1,409 rows, churn rate 26.5 % in each |
| Features | 3 standardised numeric + 37 one-hot = **40** inputs |
| Leakage controls | fit on train only · disjoint-split assertion · test hidden until notebook 07 |